In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng
!pip install -q librosa soundfile jiwer hazm
!pip install -q coqui-tts
!apt-get update -y
!espeak-ng --version
!apt-get update
!pip install -q librosa soundfile pandas numpy tqdm jiwer hazm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 58 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.

# Kamtera/persian-tts-female-vits

In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download(
    repo_id="Kamtera/persian-tts-female-vits"
)
print(MODEL_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--Kamtera--persian-tts-female-vits/snapshots/a1deb794b324844ff65c020e7a68029d99c9dd62


In [ ]:
import json
import torch
import sys
from pathlib import Path

import TTS
from TTS.tts.models.vits import Vits
from TTS.tts.configs.vits_config import VitsConfig

import os
import time
import pandas as pd
import numpy as np
import soundfile as sf

from tqdm import tqdm
from jiwer import wer, cer

import hazm

from TTS.utils.audio import AudioProcessor
from TTS.config import load_config
from TTS.api import TTS

import re


In [ ]:
model_dir = Path(MODEL_DIR)

for f in model_dir.iterdir():
    print(f.name)

print(sys.version)

with open(f"{MODEL_DIR}/config.json", "r") as f:
    cfg = json.load(f)

print(cfg.keys())

print(cfg.get("model"))
print(cfg.get("model_args"))
print(cfg.get("audio"))


ckpt = torch.load(
    f"{MODEL_DIR}/best_model_30824.pth",
    map_location="cpu"
)

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

README.md
events.out.tfevents.1673867138.b9079279db4a
phoneme_cache.zip
config-1.json
checkpoint_26000.pth
events.out.tfevents.1673900005.d9ec03af73c0
best_model_3796.pth
checkpoint_5000.pth
config.json
train_vits-1.py
.gitattributes
best_model_23604.pth
config-2.json
best_model_30824.pth
train_vits.py
config-0.json
train_vits-0.py
events.out.tfevents.1673758534.8f6deb5c5186
best_model_1725.pth
checkpoint_48000.pth
events.out.tfevents.1673947893.64d56a8e8866
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
dict_keys(['output_path', 'logger_uri', 'run_name', 'project_name', 'run_description', 'print_step', 'plot_step', 'model_param_stats', 'wandb_entity', 'dashboard_logger', 'log_model_step', 'save_step', 'save_n_checkpoints', 'save_checkpoints', 'save_all_best', 'save_best_after', 'target_loss', 'print_eval', 'test_delay_epochs', 'run_eval', 'run_eval_steps', 'distributed_backend', 'distributed_url', 'mixed_precision', 'epochs', 'batch_size', 'eval_batch_size', 'grad_clip', 'schedule

In [ ]:
CONFIG_PATH = f"{MODEL_DIR}/config.json"

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

print(cfg["model"])
print(cfg["audio"])

vits
{'fft_size': 1024, 'sample_rate': 24000, 'win_length': 1024, 'hop_length': 256, 'num_mels': 80, 'mel_fmin': 0, 'mel_fmax': None}


In [ ]:
config = VitsConfig()

config.load_json(f"{MODEL_DIR}/config.json")

In [ ]:
model = Vits.init_from_config(config)

ckpt = torch.load(
    f"{MODEL_DIR}/best_model_30824.pth",
    map_location="cpu"
)

model.load_state_dict(ckpt["model"])
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("MODEL READY")

MODEL READY


In [ ]:
[m for m in dir(model) if "infer" in m or "tts" in m or "speak" in m or "generate" in m]

['_get_speaker_id_or_dvector',
 '_init_speaker_embedding',
 '_set_speaker_input',
 'embedded_speaker_dim',
 'inference',
 'inference_noise_scale',
 'inference_noise_scale_dp',
 'inference_onnx',
 'inference_voice_conversion',
 'init_multispeaker',
 'max_inference_len',
 'num_speakers',
 'speaker_manager']

In [ ]:
print(model)

Vits(
  (text_encoder): TextEncoder(
    (emb): Embedding(156, 192)
    (encoder): RelativePositionTransformer(
      (dropout): Dropout(p=0.1, inplace=False)
      (attn_layers): ModuleList(
        (0-5): 6 x RelativePositionMultiHeadAttention(
          (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_1): ModuleList(
        (0-5): 6 x LayerNorm2()
      )
      (ffn_layers): ModuleList(
        (0-5): 6 x FeedForwardNetwork(
          (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
          (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_2): ModuleList(
        (0-5): 6 x LayerNo

In [ ]:
BENCH_PATH = "/content/drive/My Drive/asr_project/bench_sub.csv"

MODEL_DIR = "/root/.cache/huggingface/hub/models--Kamtera--persian-tts-female-vits/snapshots/a1deb794b324844ff65c020e7a68029d99c9dd62"  # CHANGE THIS
CONFIG_PATH = f"{MODEL_DIR}/config.json"
CHECKPOINT_PATH = f"{MODEL_DIR}/best_model_30824.pth"

OUTPUT_DIR = "/content/drive/My Drive/tts_benchmark/kamtera_vits"
WAV_DIR = f"{OUTPUT_DIR}/wavs"

os.makedirs(WAV_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(BENCH_PATH)
df = df.dropna(subset=["audio_path", "sentence"]).reset_index(drop=True)

print("Samples:", len(df))

Samples: 1052


In [ ]:
config = load_config(CONFIG_PATH)

model = Vits.init_from_config(config)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(checkpoint["model"])

model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [ ]:
normalizer = hazm.Normalizer()

def normalize(text):
    return normalizer.normalize(str(text)).strip()

In [ ]:
config = load_config(CONFIG_PATH)

model = Vits.init_from_config(config)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(checkpoint["model"])

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Vits(
  (text_encoder): TextEncoder(
    (emb): Embedding(156, 192)
    (encoder): RelativePositionTransformer(
      (dropout): Dropout(p=0.1, inplace=False)
      (attn_layers): ModuleList(
        (0-5): 6 x RelativePositionMultiHeadAttention(
          (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_1): ModuleList(
        (0-5): 6 x LayerNorm2()
      )
      (ffn_layers): ModuleList(
        (0-5): 6 x FeedForwardNetwork(
          (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
          (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_2): ModuleList(
        (0-5): 6 x LayerNo

In [ ]:
[x for x in dir(model) if "tts" in x.lower() or "infer" in x.lower() or "synth" in x.lower()]

['inference',
 'inference_noise_scale',
 'inference_noise_scale_dp',
 'inference_onnx',
 'inference_voice_conversion',
 'max_inference_len',
 'synthesize']

In [ ]:
print([x for x in dir(model) if "infer" in x or "tts" in x or "synth" in x])

['inference', 'inference_noise_scale', 'inference_noise_scale_dp', 'inference_onnx', 'inference_voice_conversion', 'max_inference_len', 'synthesize']


In [ ]:
def save_wav(wav, path, sr=24000):

    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().numpy()

    wav = np.array(wav).squeeze()

    sf.write(path, wav, sr)

In [ ]:
print(model.text_encoder)
print('###############################')
print(model.tokenizer if hasattr(model, "tokenizer") else "NO TOKENIZER")

TextEncoder(
  (emb): Embedding(156, 192)
  (encoder): RelativePositionTransformer(
    (dropout): Dropout(p=0.1, inplace=False)
    (attn_layers): ModuleList(
      (0-5): 6 x RelativePositionMultiHeadAttention(
        (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
        (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
        (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
        (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm_layers_1): ModuleList(
      (0-5): 6 x LayerNorm2()
    )
    (ffn_layers): ModuleList(
      (0-5): 6 x FeedForwardNetwork(
        (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
        (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,))
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm_layers_2): ModuleList(
      (0-5): 6 x LayerNorm2()
    )
  )
  (proj): Conv1d(192, 384, kernel_size=(1,), stride=(1,)

In [ ]:
def synthesize(text):
    with torch.no_grad():
        wav = model.inference([text])[0]

    if torch.is_tensor(wav):
        wav = wav.squeeze().cpu().numpy()

    return wav


def save_wav(wav, path, sr=24000):
    wav = np.asarray(wav)

    # FIX: ensure shape is (T,)
    wav = wav.reshape(-1)

    sf.write(path, wav, sr)

In [ ]:
text = "جسد مزبور را در بیشهای که در گودی کوهستان قرار دارد پیدا کردم"

wav = synthesize(text)

print(type(wav))
print(wav.shape)

import soundfile as sf
sf.write("/content/test.wav", wav, 24000)

print("saved")

AttributeError: 'list' object has no attribute 'shape'

In [ ]:
tts = TTS(model_path=CHECKPOINT_PATH, config_path=CONFIG_PATH, gpu=True)

/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


In [ ]:
print(type(model))
print(model)

<class 'TTS.tts.models.vits.Vits'>
Vits(
  (text_encoder): TextEncoder(
    (emb): Embedding(156, 192)
    (encoder): RelativePositionTransformer(
      (dropout): Dropout(p=0.1, inplace=False)
      (attn_layers): ModuleList(
        (0-5): 6 x RelativePositionMultiHeadAttention(
          (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_1): ModuleList(
        (0-5): 6 x LayerNorm2()
      )
      (ffn_layers): ModuleList(
        (0-5): 6 x FeedForwardNetwork(
          (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
          (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_2): Mod

In [ ]:
print([m for m in dir(model) if "forward" in m or "tts" in m or "infer" in m])

['_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_slow_forward', 'forward', 'forward_mas', 'inference', 'inference_noise_scale', 'inference_noise_scale_dp', 'inference_onnx', 'inference_voice_conversion', 'max_inference_len', 'register_forward_hook', 'register_forward_pre_hook']


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tts.to(device)

TTS(
  (synthesizer): Synthesizer(
    (tts_model): Vits(
      (text_encoder): TextEncoder(
        (emb): Embedding(156, 192)
        (encoder): RelativePositionTransformer(
          (dropout): Dropout(p=0.1, inplace=False)
          (attn_layers): ModuleList(
            (0-5): 6 x RelativePositionMultiHeadAttention(
              (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
              (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
              (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
              (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (norm_layers_1): ModuleList(
            (0-5): 6 x LayerNorm2()
          )
          (ffn_layers): ModuleList(
            (0-5): 6 x FeedForwardNetwork(
              (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
              (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,)

In [ ]:
text = "جسد مزبور را در بیشهای که در گودی کوهستان قرار دارد پیدا کردم"

wav = tts.tts(text)

In [ ]:
wav = np.array(wav)
wav = wav.reshape(-1)

sf.write("/content/test.wav", wav, 24000)

print("saved")

saved


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/tts_benchmark/vits_coqui"
WAV_DIR = os.path.join(OUTPUT_DIR, "wavs")

os.makedirs(WAV_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/asr_project/bench_sub.csv")
df = df.dropna().reset_index(drop=True)

print("Samples:", len(df))

Samples: 1052


In [ ]:
normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize(text):
    text = str(text)
    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
import os
import time
import numpy as np
import soundfile as sf
from tqdm import tqdm
import librosa

pred_texts = []
ref_texts = []
audio_paths = []
tts_runtimes = []
audio_durations = []

SAMPLE_RATE = 24000

total_audio_duration = 0.0
start_time = time.time()

os.makedirs(WAV_DIR, exist_ok=True)

for i, row in tqdm(df.iterrows(), total=len(df)):

    text = normalize(row["sentence"])
    ref_texts.append(text)

    out_path = os.path.join(WAV_DIR, f"{i:05d}.wav")

    try:
        t0 = time.time()

        wav = tts.tts(text)

        # flatten safely
        wav = np.array(wav).reshape(-1)

        t1 = time.time()

        # save audio
        sf.write(out_path, wav, SAMPLE_RATE)

        # ---- duration (IMPORTANT for RTF) ----
        duration = len(wav) / SAMPLE_RATE
        audio_durations.append(duration)

        total_audio_duration += duration

        tts_runtimes.append(t1 - t0)

        audio_paths.append(out_path)

    except Exception as e:
        print("FAILED:", i, e)
        continue

100%|██████████| 1052/1052 [03:03<00:00,  5.72it/s]


In [ ]:
import pandas as pd

tts_df = pd.DataFrame({
    "audio_path": audio_paths,
    "reference": ref_texts,
    "tts_runtime": tts_runtimes,
    "audio_duration": audio_durations
})

SAVE_PATH = "/content/tts_outputs.csv"
tts_df.to_csv(SAVE_PATH, index=False)

print("Saved:", SAVE_PATH)

Saved: /content/tts_outputs.csv


In [ ]:
pred_texts = []
ref_texts = []
audio_paths = []
runtimes = []
audio_durations = []
rtf_list = []

start_time = time.time()

for i, row in tqdm(df.iterrows(), total=len(df)):

    text = normalize(row["sentence"])
    ref_texts.append(text)

    out_path = os.path.join(WAV_DIR, f"{i:05d}.wav")

    try:
        t0 = time.time()

        wav = tts.tts(text)

        wav = np.array(wav).reshape(-1)

        # 🔊 compute audio duration (IMPORTANT)
        duration = len(wav) / 24000.0   # sample rate = 24kHz
        audio_durations.append(duration)

        sf.write(out_path, wav, 24000)

        t1 = time.time()

        runtime = t1 - t0
        runtimes.append(runtime)

        # 🔥 RTF per sample
        rtf_list.append(runtime / duration)

        audio_paths.append(out_path)

    except Exception as e:
        print("FAILED:", i, e)
        continue

100%|██████████| 1052/1052 [10:48<00:00,  1.62it/s]


In [ ]:
import re
import hazm

normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):
    text = str(text)
    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    text = re.sub(r"[^\w\s]", " ", text)
    text = text.replace("می ", "می")
    text = text.replace("نمی ", "نمی")
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
from jiwer import wer, cer
from tqdm import tqdm
import time

predictions = []
references = []
asr_runtimes = []
audio_durations = df["audio_duration"].tolist()

total_audio_duration = sum(audio_durations)

start_time = time.time()

for i, row in tqdm(df.iterrows(), total=len(df)):

    audio_path = row["audio_path"]

    try:
        t0 = time.time()

        pred = asr_model.transcribe([audio_path], batch_size=1)[0]

        t1 = time.time()

        pred = normalize_persian(pred)
        ref = normalize_persian(row["reference"])

        predictions.append(pred)
        references.append(ref)
        asr_runtimes.append(t1 - t0)

    except Exception as e:
        print("FAILED:", audio_path, e)
        continue

KeyError: 'audio_duration'

In [ ]:
pred_texts = []
ref_texts = []
audio_paths = []
runtimes = []
audio_durations = []
rtf_list = []

start_time = time.time()

for i, row in tqdm(df.iterrows(), total=len(df)):

    text = normalize(row["sentence"])
    ref_texts.append(text)

    out_path = os.path.join(WAV_DIR, f"{i:05d}.wav")

    try:
        t0 = time.time()

        wav = tts.tts(text)

        wav = np.array(wav).reshape(-1)

        # 🔊 compute audio duration (IMPORTANT)
        duration = len(wav) / 24000.0   # sample rate = 24kHz
        audio_durations.append(duration)

        sf.write(out_path, wav, 24000)

        t1 = time.time()

        runtime = t1 - t0
        runtimes.append(runtime)

        # 🔥 RTF per sample
        rtf_list.append(runtime / duration)

        audio_paths.append(out_path)

    except Exception as e:
        print("FAILED:", i, e)
        continue

 45%|████▌     | 474/1052 [01:23<02:25,  3.98it/s]

In [ ]:
total_time = sum(runtimes)
total_audio = sum(audio_durations)

global_rtf = total_time / total_audio

print("TTS RTF:", global_rtf)

TTS RTF: 0.1509917050572368


In [ ]:
import pandas as pd

df_out = pd.DataFrame({
    "audio_path": audio_paths,
    "text": ref_texts,
    "tts_runtime": runtimes,
    "audio_duration": audio_durations,
    "tts_rtf": rtf_list
})

df_out.to_csv(os.path.join(WAV_DIR, "tts_dataset.csv"), index=False)

In [ ]:
import pandas as pd

df_out = pd.DataFrame({
    "audio_path": audio_paths,
    "text": ref_texts,
    "tts_runtime": runtimes,
    "audio_duration": audio_durations,
    "tts_rtf": rtf_list
})

df_out.to_csv(os.path.join(WAV_DIR, "tts_dataset.csv"), index=False)

In [ ]:
df_out = pd.DataFrame({
    "audio_path": audio_paths,
    "text": ref_texts,
    "runtime": runtimes
})

df_out.to_csv("/content/tts_dataset.csv", index=False)

In [ ]:
wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

NameError: name 'references' is not defined

In [ ]:
total_asr_time = sum(asr_runtimes)

asr_rtf = total_asr_time / total_audio_duration
avg_asr_latency = total_asr_time / len(predictions)

In [ ]:
total_tts_time = sum(df["tts_runtime"])
total_tts_audio = sum(df["audio_duration"])

tts_rtf = total_tts_time / total_tts_audio
avg_tts_latency = total_tts_time / len(df)

In [ ]:
print("\n===== TTS + ASR BENCHMARK =====\n")

print("ASR WER:", wer_score)
print("ASR CER:", cer_score)

print("ASR RTF:", asr_rtf)
print("ASR latency:", avg_asr_latency)

print("TTS RTF:", tts_rtf)
print("TTS latency:", avg_tts_latency)

print("\nTotal audio duration:", total_audio_duration)
print("Total ASR time:", total_asr_time)
print("Total TTS time:", total_tts_time)